In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
print(os.listdir('/content/drive/MyDrive'))

['Colab Notebooks', '310624150036 Sandhiya M', 'Classroom', '17575105002242808708631139777670.jpg', 'DOC-20251207-WA0002._copy.pdf', '17765191739047810477415598380488.jpg', '17765192150912454281606117512065.jpg', '17765192309143195835960957734967.jpg', 'IMG_20260418_190938.jpg', '17766099971848538965030527918766.jpg', 'IMG-20250401-WA0005.jpg', '1776610978587822859449833959454.jpg', '17766112252923561344807022211291.jpg', '17766112653621321055412558166154.jpg', '17766114143313916421967245330503.jpg', '17766114769056416460758982676037.jpg', 'IMG_20260419_210441.jpg', 'Document from Sandhiya (1).pdf', 'Document from Sandhiya', 'Document from Sandhiya.pdf', 'nature_12K.zip']


In [ ]:
import zipfile

zip_path = "/content/drive/MyDrive/nature_12K.zip"
extract_path = "/content/inaturalist"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")

Dataset extracted successfully!


In [ ]:
import os
print(os.listdir("/content/inaturalist"))

['inaturalist_12K']


In [ ]:
import os

dataset_path = "/content/inaturalist/inaturalist_12K"

print("Folders:", os.listdir(dataset_path))

Folders: ['val', 'train', '.DS_Store']


In [ ]:
import os

train_path = "/content/inaturalist/inaturalist_12K/train"
val_path = "/content/inaturalist/inaturalist_12K/val"

# Ignore hidden files like .DS_Store
train_classes = sorted([c for c in os.listdir(train_path) if not c.startswith(".")])
val_classes = sorted([c for c in os.listdir(val_path) if not c.startswith(".")])

print("Train classes:", len(train_classes))
print(train_classes)

print("\nValidation classes:", len(val_classes))
print(val_classes)

Train classes: 10
['Amphibia', 'Animalia', 'Arachnida', 'Aves', 'Fungi', 'Insecta', 'Mammalia', 'Mollusca', 'Plantae', 'Reptilia']

Validation classes: 10
['Amphibia', 'Animalia', 'Arachnida', 'Aves', 'Fungi', 'Insecta', 'Mammalia', 'Mollusca', 'Plantae', 'Reptilia']


In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Image transformations
train_transform
= transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Dataset paths
train_dir = "/content/inaturalist/inaturalist_12K/train"
val_dir = "/content/inaturalist/inaturalist_12K/val"

# Load datasets
train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
val_dataset = datasets.ImageFolder(val_dir, transform=val_transform)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

print("Train images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Classes:", train_dataset.classes)

Train images: 9999
Validation images: 2000
Classes: ['Amphibia', 'Animalia', 'Arachnida', 'Aves', 'Fungi', 'Insecta', 'Mammalia', 'Mollusca', 'Plantae', 'Reptilia']


In [ ]:
import torch
import torch.nn as nn

class CNNModel(nn.Module):
    def __init__(self, activation="relu", dropout=0.2):
        super().__init__()

        # Activation function
        activations = {
            "relu": nn.ReLU(),
            "gelu": nn.GELU(),
            "silu": nn.SiLU(),
            "mish": nn.Mish()
        }
        act = activations[activation]

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            act,
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            act,
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            act,
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            act,
            nn.MaxPool2d(2),

            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            act,
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(512 * 7 * 7, 128),
            act,
            nn.Dropout(dropout),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Create model
model = CNNModel(activation="relu", dropout=0.2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(model)
print("Device:", device)

In [ ]:
images, labels = next(iter(train_loader))

images = images.to(device)

outputs = model(images)

print("Input batch shape :", images.shape)
print("Output batch shape:", outputs.shape)

In [ ]:
import wandb

wandb.login()

In [ ]:
import torch.optim as optim

# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Loss Function :", criterion)
print("Optimizer :", optimizer)

In [ ]:

def train_one_epoch(model, train_loader, optimizer, criterion, device):
    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc = 100 * correct / total

    return train_loss, train_acc

In [ ]:

def validate(model, val_loader, criterion, device):
    model.eval()

    running_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()

            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    val_loss = running_loss / len(val_loader)
    val_acc = 100 * correct / total

    return val_loss, val_acc

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,      # changed from 32
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

In [ ]:
images, labels = next(iter(train_loader))

print("Images:", images.shape)
print("Labels:", labels.shape)
print("Device:", device)

In [ ]:
print(train_one_epoch)
print(validate)

In [ ]:
epochs = 5

for epoch in range(epochs):

    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )

    val_loss, val_acc = validate(
        model, val_loader, criterion, device
    )

    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Accuracy: {train_acc:.2f}%")
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Validation Accuracy: {val_acc:.2f}%")
    print("-" * 40)

In [ ]:
import torch

print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Still using CPU")

In [ ]:
import wandb

sweep_config = {
    "method": "random",

    "metric": {
        "name": "val_accuracy",
        "goal": "maximize"
    },

    "parameters": {
        "learning_rate": {
            "values": [0.001, 0.0005, 0.0001]
        },
        "batch_size": {
            "values": [16, 32]
        },
        "dropout": {
            "values": [0.2, 0.3]
        },
        "activation": {
            "values": ["relu", "gelu"]
        }
    }
}

In [ ]:
sweep_id = wandb.sweep(
    sweep=sweep_config,
    project="cs6910-assignment-2"
)

print("Sweep ID:", sweep_id)

In [ ]:
def sweep_train():
    run = wandb.init()

    config = wandb.config

    # Create datasets with the selected batch size
    train_loader = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    # Create model using sweep hyperparameters
    model = CNNModel(
        activation=config.activation,
        dropout=config.dropout
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config.learning_rate
    )

    criterion = nn.CrossEntropyLoss()

    epochs = 3   # Keep 3 epochs for sweep to save time

    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, criterion, device
        )

        val_loss, val_acc = validate(
            model, val_loader, criterion, device
        )

        wandb.log({
             "epoch": epoch + 1,
             "train_loss": train_loss,
             "train_accuracy": train_acc,
             "val_loss": val_loss,
             "val_accuracy": val_acc
        })
    run.finish()


In [ ]:
wandb.agent(
    sweep_id,
    function=sweep_train,
    count=8
)